# Dataset Preparation

This notebook prepares the datasets for the cross-dataset facial emotion recognition experiment.

The main source dataset is KMU-FED. The model will first be trained and tested on KMU-FED to obtain the baseline. Then the same trained model will be tested on KDEF and FER2013 without retraining.

The shared emotion classes are:

- anger
- disgust
- fear
- happy
- sadness
- surprise

Neutral is excluded because KMU-FED contains six basic expressions, while KDEF and FER2013 also include neutral.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter

import pandas as pd
from sklearn.model_selection import train_test_split

## Project Paths

The raw datasets are stored in Google Drive. The generated manifest CSV files are saved under `data_processed/manifests`.

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/FER_Generalization_Project")
RAW_DIR = BASE_DIR / "data_raw"
PROCESSED_DIR = BASE_DIR / "data_processed"
MANIFEST_DIR = PROCESSED_DIR / "manifests"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

KMU_PATH = RAW_DIR / "KMU_FED" / "KMU-FED"
KDEF_PATH = RAW_DIR / "KDEF" / "KDEF_and_AKDEF" / "KDEF_and_AKDEF" / "KDEF"
FER2013_PATH = RAW_DIR / "FER2013" / "archive (1)"

print("KMU_PATH:", KMU_PATH.exists(), KMU_PATH)
print("KDEF_PATH:", KDEF_PATH.exists(), KDEF_PATH)
print("FER2013_PATH:", FER2013_PATH.exists(), FER2013_PATH)

## Shared Classes and Label Alignment

All datasets are mapped to six common emotion classes. Dataset-specific label codes are converted into the same final class names.

In [ ]:
CLASSES = ["anger", "disgust", "fear", "happy", "sadness", "surprise"]

class_to_idx = {label: idx for idx, label in enumerate(CLASSES)}

kmu_label_map = {
    "AN": "anger",
    "DI": "disgust",
    "FE": "fear",
    "HA": "happy",
    "SA": "sadness",
    "SU": "surprise",
}

kdef_label_map = {
    "AF": "fear",
    "AN": "anger",
    "DI": "disgust",
    "HA": "happy",
    "NE": "neutral",
    "SA": "sadness",
    "SU": "surprise",
}

fer_label_map = {
    "angry": "anger",
    "disgust": "disgust",
    "fear": "fear",
    "happy": "happy",
    "sad": "sadness",
    "surprise": "surprise",
    "neutral": "neutral",
}

print("Classes:", CLASSES)
print("Class to index:", class_to_idx)

## Manifest Creation

A manifest is a CSV file containing the image path, emotion label, label index, and dataset name.

KMU-FED is split into 80% training and 20% testing using stratified sampling. KDEF and FER2013 are prepared as external test datasets.

In [ ]:
print("Manifest files created:")

for file in MANIFEST_DIR.glob("*.csv"):
    print(file.name)

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path("/content/drive/MyDrive/FER_Generalization_Project")
RAW_DIR = BASE_DIR / "data_raw"
PROCESSED_DIR = BASE_DIR / "data_processed"
MANIFEST_DIR = PROCESSED_DIR / "manifests"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

KMU_PATH = RAW_DIR / "KMU_FED" / "KMU-FED"
KDEF_PATH = RAW_DIR / "KDEF" / "KDEF_and_AKDEF" / "KDEF_and_AKDEF" / "KDEF"
FER2013_PATH = RAW_DIR / "FER2013" / "archive (1)"

print("KMU_PATH:", KMU_PATH.exists(), KMU_PATH)
print("KDEF_PATH:", KDEF_PATH.exists(), KDEF_PATH)
print("FER2013_PATH:", FER2013_PATH.exists(), FER2013_PATH)

# -----------------------------
# Shared classes
# -----------------------------
CLASSES = ["anger", "disgust", "fear", "happy", "sadness", "surprise"]
class_to_idx = {label: idx for idx, label in enumerate(CLASSES)}

# -----------------------------
# Label maps
# -----------------------------
kmu_label_map = {
    "AN": "anger",
    "DI": "disgust",
    "FE": "fear",
    "HA": "happy",
    "SA": "sadness",
    "SU": "surprise",
}

kdef_label_map = {
    "AF": "fear",
    "AN": "anger",
    "DI": "disgust",
    "HA": "happy",
    "NE": "neutral",
    "SA": "sadness",
    "SU": "surprise",
}

fer_label_map = {
    "angry": "anger",
    "disgust": "disgust",
    "fear": "fear",
    "happy": "happy",
    "sad": "sadness",
    "surprise": "surprise",
    "neutral": "neutral",
}

# -----------------------------
# KMU-FED manifest
# -----------------------------
kmu_rows = []

for img_path in KMU_PATH.glob("*.jpg"):
    parts = img_path.stem.split("_")

    if len(parts) < 2:
        continue

    emotion_code = parts[1]
    label = kmu_label_map.get(emotion_code)

    if label in CLASSES:
        kmu_rows.append({
            "path": str(img_path),
            "label": label,
            "label_idx": class_to_idx[label],
            "dataset": "KMU_FED"
        })

kmu_df = pd.DataFrame(kmu_rows)

kmu_train_df, kmu_test_df = train_test_split(
    kmu_df,
    test_size=0.20,
    random_state=42,
    stratify=kmu_df["label"]
)

kmu_train_df = kmu_train_df.reset_index(drop=True)
kmu_test_df = kmu_test_df.reset_index(drop=True)

# -----------------------------
# KDEF frontal-only manifest
# -----------------------------
kdef_rows = []

for img_path in KDEF_PATH.rglob("*.JPG"):
    name = img_path.stem

    if len(name) < 7:
        continue

    emotion_code = name[4:6]
    view_code = name[6:]

    label = kdef_label_map.get(emotion_code)

    if view_code == "S" and label in CLASSES:
        kdef_rows.append({
            "path": str(img_path),
            "label": label,
            "label_idx": class_to_idx[label],
            "dataset": "KDEF"
        })

kdef_df = pd.DataFrame(kdef_rows)

# -----------------------------
# FER2013 test-only manifest
# -----------------------------
fer_rows = []
fer_test_path = FER2013_PATH / "test"

for class_folder in fer_test_path.iterdir():
    if not class_folder.is_dir():
        continue

    original_label = class_folder.name
    label = fer_label_map.get(original_label)

    if label not in CLASSES:
        continue

    for img_path in class_folder.glob("*.jpg"):
        fer_rows.append({
            "path": str(img_path),
            "label": label,
            "label_idx": class_to_idx[label],
            "dataset": "FER2013"
        })

fer_df = pd.DataFrame(fer_rows)

# -----------------------------
# Save CSV files
# -----------------------------
kmu_df.to_csv(MANIFEST_DIR / "kmu_all.csv", index=False)
kmu_train_df.to_csv(MANIFEST_DIR / "kmu_train.csv", index=False)
kmu_test_df.to_csv(MANIFEST_DIR / "kmu_test.csv", index=False)
kdef_df.to_csv(MANIFEST_DIR / "kdef_frontal_test.csv", index=False)
fer_df.to_csv(MANIFEST_DIR / "fer2013_test.csv", index=False)

print("\nSaved manifest files:")
for file in MANIFEST_DIR.glob("*.csv"):
    print(file.name)

print("\nKMU train:", len(kmu_train_df))
print("KMU test:", len(kmu_test_df))
print("KDEF test:", len(kdef_df))
print("FER2013 test:", len(fer_df))

## Manifest Summary

This section verifies the number of images per dataset and per class.

In [ ]:
print("Manifest files available:")

for file in MANIFEST_DIR.glob("*.csv"):
    print(file.name)

In [ ]:
kmu_train_df = pd.read_csv(MANIFEST_DIR / "kmu_train.csv")
kmu_test_df = pd.read_csv(MANIFEST_DIR / "kmu_test.csv")
kdef_test_df = pd.read_csv(MANIFEST_DIR / "kdef_frontal_test.csv")
fer2013_test_df = pd.read_csv(MANIFEST_DIR / "fer2013_test.csv")

print("KMU train:", len(kmu_train_df))
print("KMU test:", len(kmu_test_df))
print("KDEF test:", len(kdef_test_df))
print("FER2013 test:", len(fer2013_test_df))

kmu_train_df.head()

In [ ]:
for file in MANIFEST_DIR.glob("*.csv"):
    df = pd.read_csv(file)

    print("\n", file.name)
    print("Total images:", len(df))
    print(df["label"].value_counts().sort_index())

## Image Loading Check

Before training, we verify that images can be loaded correctly using a PyTorch Dataset and DataLoader.

In [ ]:
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
#Image transforamtion
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# dataset class
class FERManifestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = row["path"]
        label = int(row["label_idx"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
BATCH_SIZE = 32

kmu_train_dataset = FERManifestDataset(kmu_train_df, transform=train_transform)
kmu_test_dataset = FERManifestDataset(kmu_test_df, transform=test_transform)

kdef_test_dataset = FERManifestDataset(kdef_test_df, transform=test_transform)
fer2013_test_dataset = FERManifestDataset(fer2013_test_df, transform=test_transform)

kmu_train_loader = DataLoader(
    kmu_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

kmu_test_loader = DataLoader(
    kmu_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

kdef_test_loader = DataLoader(
    kdef_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

fer2013_test_loader = DataLoader(
    fer2013_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Dataloaders ready.")

In [ ]:
images, labels = next(iter(kmu_train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First labels:", labels[:10])

In [ ]:
def unnormalize_image(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    image = tensor.cpu() * std + mean
    image = torch.clamp(image, 0, 1)

    return image.permute(1, 2, 0)

images, labels = next(iter(kmu_train_loader))

plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(unnormalize_image(images[i]))
    plt.title(CLASSES[labels[i].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

## Dataset Preparation Completed

The dataset preparation step is complete.

Final dataset setup:

- KMU-FED train set: 884 images
- KMU-FED test set: 222 images
- KDEF frontal test set: 840 images
- FER2013 test set: 5945 images

The six aligned emotion classes are:

- anger
- disgust
- fear
- happy
- sadness
- surprise

Neutral was excluded because KMU-FED contains six basic expressions. The next step is to train a pretrained SqueezeNet 1.1 model on KMU-FED and evaluate it first on the KMU-FED test set to obtain the baseline.